# Notebook 07 — Failure Analysis

**Learning objectives:**
- Understand which BindCraft filters reject the most designs
- Compare metric distributions of accepted vs. rejected designs
- Interpret failure patterns to improve next design rounds
- Learn to find signal in negative results

**Time:** ~45 minutes

**Companion wiki page:** [5.4 Filtering](https://rucha1796.github.io/internship-bindcraft-2026/m5_04_filtering/)

---
> No GPU required.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

DATA_DIR = "/content/drive/MyDrive/bindcraft/PDL1_8aok"

accepted_csv = f"{DATA_DIR}/final_design_stats.csv"
failed_csv   = f"{DATA_DIR}/failure_csv.csv"
traj_csv     = f"{DATA_DIR}/trajectory_stats.csv"

KEY_METRICS = [
    "Average_i_pTM", "Average_pLDDT", "Average_Binder_pLDDT",
    "Average_Binder_RMSD", "Average_ShapeComplementarity",
    "Average_dG", "Average_i_pAE", "Average_Relaxed_Clashes"
]

# Load datasets
def load_or_demo(path, demo_fn, label):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✓ Loaded {label}: {len(df)} rows")
        return df
    else:
        print(f"  {label} not found — generating demo data")
        return demo_fn()

np.random.seed(42)

def demo_accepted():
    n = 12
    return pd.DataFrame({
        "binder_name": [f"PDL1_t{i}_mpnn1" for i in range(1, n+1)],
        **{m: np.random.uniform(*([0.56,0.74] if "pTM" in m else [0.86,0.94] if "pLDDT" in m
                                    and "Binder" not in m else [0.81,0.91] if "Binder_p" in m
                                    else [0.5,1.9] if "RMSD" in m else [0.56,0.72] if "Shape" in m
                                    else [-24,-11] if "dG" in m else [5,9.5] if "pAE" in m
                                    else [0,4.5]), n)
           for m in KEY_METRICS}
    })

def demo_failed():
    n = 40
    df = pd.DataFrame({
        "binder_name": [f"PDL1_t{i}_mpnn1" for i in range(20, 20+n)],
        **{m: np.random.uniform(*([0.2,0.58] if "pTM" in m else [0.7,0.87] if "pLDDT" in m
                                    and "Binder" not in m else [0.6,0.82] if "Binder_p" in m
                                    else [1.0,4.5] if "RMSD" in m else [0.3,0.58] if "Shape" in m
                                    else [-15,-3] if "dG" in m else [7,18] if "pAE" in m
                                    else [2,15]), n)
           for m in KEY_METRICS}
    })
    # Assign filter failure reasons
    fail_reasons = np.random.choice(
        ["Average_i_pTM", "Average_Binder_RMSD", "Average_ShapeComplementarity",
         "Average_dG", "Average_pLDDT"], size=n
    )
    df["failed_filter"] = fail_reasons
    return df

df_accepted = load_or_demo(accepted_csv, demo_accepted, "accepted designs")
df_failed   = load_or_demo(failed_csv, demo_failed, "failed designs")

## Cell 2 — Run statistics

In [ ]:
n_accepted = len(df_accepted)
n_failed   = len(df_failed)
n_total    = n_accepted + n_failed
accept_rate = n_accepted / n_total * 100 if n_total > 0 else 0

print("=" * 50)
print("RUN STATISTICS")
print("=" * 50)
print(f"Total trajectories/MPNN designs: {n_total}")
print(f"Accepted (passed all filters):   {n_accepted}  ({accept_rate:.1f}%)")
print(f"Rejected (failed ≥1 filter):     {n_failed}  ({100-accept_rate:.1f}%)")
print()
print("Interpretation:")
if accept_rate >= 20:
    print(f"  ✓ {accept_rate:.0f}% acceptance rate is good for de novo binder design.")
elif accept_rate >= 5:
    print(f"  ~ {accept_rate:.0f}% acceptance rate is typical. More trajectories = more designs.")
else:
    print(f"  ⚠ {accept_rate:.0f}% acceptance rate is low. Consider adjusting hotspots or filter preset.")

## Cell 3 — Which filter fails most?

In [ ]:
if "failed_filter" in df_failed.columns:
    failure_counts = df_failed["failed_filter"].value_counts()
    
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["#c0392b" if "i_pTM" in x else "#e67e22" if "RMSD" in x
              else "#f39c12" if "Shape" in x else "#3498db" if "pLDDT" in x
              else "#95a5a6" for x in failure_counts.index]
    
    failure_counts.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_xlabel("Number of designs rejected by this filter", fontsize=11)
    ax.set_title("Which filter rejects the most designs?", fontsize=12)
    ax.axvline(failure_counts.max() * 0.5, color="gray", linestyle="--", alpha=0.4)
    
    for i, (val, name) in enumerate(zip(failure_counts.values, failure_counts.index)):
        ax.text(val + 0.3, i, f"{val}", va="center", fontsize=10)
    
    plt.tight_layout()
    plt.savefig("failure_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print("\nInterpretation guide:")
    most_common = failure_counts.index[0]
    print(f"  Most common failure: {most_common}")
    
    guidance = {
        "Average_i_pTM": "Hallucination struggles with this target. Try different hotspot residues or more trajectories.",
        "Average_Binder_RMSD": "Designs aren't self-folding. Try shorter binder lengths or different helicity settings.",
        "Average_ShapeComplementarity": "Surface geometry isn't matching. The target surface may be difficult to design against.",
        "Average_dG": "Predicted binding energy not favorable enough. May improve with more trajectories.",
        "Average_pLDDT": "Binder structural confidence low. Try more recycles or different length range."
    }
    print(f"  Suggestion: {guidance.get(most_common, 'Investigate this metric further.')}")
else:
    print("'failed_filter' column not in CSV. The full run data includes this column.")
    print("Showing overall statistics instead.")
    
    # Infer failures from threshold violations
    THRESHOLDS = {
        "Average_i_pTM": ("<", 0.55),
        "Average_Binder_RMSD": (">", 2.0),
        "Average_ShapeComplementarity": ("<", 0.55),
        "Average_dG": (">", -10),
        "Average_pLDDT": ("<", 0.85)
    }
    print("\nMetrics below threshold in REJECTED designs (estimated):")
    for col, (op, threshold) in THRESHOLDS.items():
        if col in df_failed.columns:
            if op == "<":
                n = (df_failed[col] < threshold).sum()
            else:
                n = (df_failed[col] > threshold).sum()
            pct = n / len(df_failed) * 100
            print(f"  {col}: {n}/{len(df_failed)} ({pct:.0f}%)")

## Cell 4 — Accepted vs. Rejected: metric comparison

In [ ]:
compare_metrics = [(m, t) for m, t in [
    ("Average_i_pTM", 0.55),
    ("Average_Binder_RMSD", 2.0),
    ("Average_ShapeComplementarity", 0.55),
    ("Average_dG", -10),
] if m in df_accepted.columns and m in df_failed.columns]

n_cols = 2
n_rows = (len(compare_metrics) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4*n_rows))
axes = axes.flatten()

for ax, (col, threshold) in zip(axes, compare_metrics):
    accepted_vals = df_accepted[col].dropna()
    failed_vals   = df_failed[col].dropna()
    
    all_vals = pd.concat([accepted_vals, failed_vals])
    bins = np.linspace(all_vals.min(), all_vals.max(), 20)
    
    ax.hist(failed_vals, bins=bins, alpha=0.6, color="#c0392b", label=f"Rejected (n={len(failed_vals)})")
    ax.hist(accepted_vals, bins=bins, alpha=0.8, color="#27ae60", label=f"Accepted (n={len(accepted_vals)})")
    ax.axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"threshold={threshold}")
    
    ax.set_title(col.replace("Average_", ""), fontsize=11)
    ax.legend(fontsize=9)

for ax in axes[len(compare_metrics):]:
    ax.set_visible(False)

plt.suptitle("Accepted vs. Rejected Designs: Metric Distributions",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("accepted_vs_rejected.png", dpi=150, bbox_inches="tight")
plt.show()

## Cell 5 — What would happen with relaxed filters?

In [ ]:
RELAXED_THRESHOLDS = {
    "Average_i_pTM":               0.45,
    "Average_pLDDT":               0.80,
    "Average_Binder_pLDDT":        0.70,
    "Average_i_pAE":               15.0,
    "Average_Binder_RMSD":         3.0,
    "Average_ShapeComplementarity": 0.45,
    "Average_dG":                  -5.0,
    "Average_Relaxed_Clashes":     10.0,
}

def passes_filters(row, thresholds):
    for col, threshold in thresholds.items():
        if col not in row:
            continue
        # Higher = better for pTM, pLDDT, ShapeComp; lower = better for RMSD, pAE, Clashes, dG
        lower_better = ["Average_Binder_RMSD", "Average_i_pAE", "Average_Relaxed_Clashes", "Average_dG"]
        if col in lower_better:
            if row[col] > threshold:
                return False
        else:
            if row[col] < threshold:
                return False
    return True

# How many rejected designs would pass with relaxed filters?
would_pass = df_failed.apply(lambda r: passes_filters(r, RELAXED_THRESHOLDS), axis=1).sum()

print("=== Filter Sensitivity Analysis ===")
print(f"Currently accepted (default filters): {len(df_accepted)}")
print(f"Currently rejected:                   {len(df_failed)}")
print(f"Would also pass with RELAXED filters: {would_pass}")
print(f"Potential total with relaxed filters: {len(df_accepted) + would_pass}")
print()
print("Implication:")
print("  Relaxed filters let through more designs, but quality is lower.")
print("  Default filters are recommended unless you have very few accepted designs.")

## Cell 6 — Recommendations for next design round

In [ ]:
print("=" * 60)
print("RECOMMENDATIONS FOR NEXT ROUND")
print("=" * 60)
print()

# Check i_pTM distribution in failed designs
if "Average_i_pTM" in df_failed.columns:
    mean_failed_iptm = df_failed["Average_i_pTM"].mean()
    print(f"Mean i_pTM of rejected designs: {mean_failed_iptm:.3f}")
    
    if mean_failed_iptm < 0.3:
        print("  → Most rejections are due to very low i_pTM.")
        print("  → The target surface may be hard to design against with default settings.")
        print("  → Suggestions: try different hotspot residues, increase trajectory count,")
        print("    or reduce binder length range.")
    elif mean_failed_iptm < 0.5:
        print("  → Moderate i_pTM in rejected designs — close to passing.")
        print("  → Increasing trajectory count (50 → 100) may yield more accepted designs.")
    else:
        print("  → Many rejected designs have decent i_pTM but fail other filters.")
        print("  → Check which non-i_pTM filter is the main bottleneck (Cell 3).")

print()

# Check Binder_RMSD distribution
if "Average_Binder_RMSD" in df_failed.columns:
    high_rmsd = (df_failed["Average_Binder_RMSD"] > 2.0).sum()
    pct = high_rmsd / len(df_failed) * 100
    print(f"{high_rmsd}/{len(df_failed)} ({pct:.0f}%) rejected designs have Binder_RMSD > 2.0 Å")
    if pct > 30:
        print("  → High Binder_RMSD failure rate. Binders aren't self-folding.")
        print("  → Suggestions: try shorter binder lengths, or adjust helicity in advanced settings.")

print()
print("General best practice:")
print("  - Run more trajectories (100-200) if time allows")
print("  - Try 2-3 different hotspot sets if i_pTM acceptance is low")
print("  - Use experimental results (when available) to guide next round")

## Questions to answer

**Q1:** What is the acceptance rate in this dataset? Is it higher or lower than you expected?

_Your answer:_

**Q2:** Which filter rejects the most designs? What does that tell you about this target?

_Your answer:_

**Q3:** Looking at the histogram comparison (Cell 4), where do rejected and accepted designs overlap? What does this "gray zone" represent?

_Your answer:_

**Q4:** If you were to do a second BindCraft run, what ONE thing would you change based on this analysis?

_Your answer:_

---
**Next:** Notebook 08 — Interactive design selection with sliders